# Airflow. Оркестрация пайплайнов

- Автор: Исмаилов Эльчин
- Дата: 29.07.2026

## Цели и задачи проекта

Сервис Яндекс Книги предоставляет доступ к контенту разных форматов, включая текст, аудио и не только. В этом проекте вы построите пайплайн в Airflow, который будет запускать PySpark-скрипт для обработки данных и создания витрин. Эти витрины помогут команде сервиса быстрее и проще готовить отчёты.

В проекте вам нужно только создать DAG для запуска уже готового Spark-кода. Вручную анализировать данные вам не понадобится.

## Описание данных

Таблица `bookmate.audition` содержит данные об активности пользователей и включает столбцы:

* `audition_id` — уникальный идентификатор сессии чтения или прослушивания;

* `puid` — идентификатор пользователя;

* `usage_platform_ru` — название платформы, с помощью которой пользователь взаимодействует с контентом;

* `msk_business_dt_str` — дата и время события (строка, часовой пояс — МСК);

* `app_version` — версия приложения;

* `adult_content_flg` — значение, которое показывает, был ли контент для взрослых (`True` или `False`);

* `hours` — длительность сессии чтения или прослушивания в часах;

* `hours_sessions_long` — длительность длинных сессий в часах;

* `kids_content_flg` — значение, которое показывает, был ли это детский контент (`True` или `False`);

* `main_content_id` — идентификатор основного контента;

* `usage_geo_id` — идентификатор географического местоположения пользователя.

Таблица `bookmate.content` включает столбцы:

* `main_content_id` — идентификатор основного контента;

* `main_author_id` — идентификатор основного автора контента;

* `main_content_type` — тип контента: аудио, текст или другой;

* `main_content_name` — название контента;

* `main_content_duration_hours` — длительность контента в часах;

* `published_topic_title_list` — список жанров или тем контента.</font>

## Содержимое проекта

Проект предполагает несколько шагов:

1. Написать Spark-код — вы подключитесь к своему хранилищу данных и укажете, куда сохранять результат.

2. Создать DAG — он будет запускать Spark-код. В DAG вы опишете задачи с помощью `PythonOperator` и `DataprocCreatePysparkJobOperator`, настроите зависимости и получите агрегированные данные для первых бизнес-выводов.

3. Запустить Airflow — именно он будет управлять вашим пайплайном.

## 1. Написание Spark-кода

In [ ]:
!pip install clickhouse-connect
from clickhouse_connect import get_client

client = get_client(
    host="DATA_DELETED",
    port=8443,
    username="DATA_DELETED",  
    password="DATA_DELETED", 
    secure=True,
    verify=False,
    database="DATA_DELETED"
)

print(client.query("SHOW TABLES").result_rows)
print(client.query("SELECT * FROM bookmate_user_aggregate LIMIT 10").result_rows)


     |████████████████████████████████| 1.1 MB 2.3 MB/s eta 0:00:01
     |████████████████████████████████| 5.6 MB 60.5 MB/s eta 0:00:01
     |████████████████████████████████| 1.4 MB 73.9 MB/s eta 0:00:01
[('bookmate_user_aggregate',), ('games_user_aggregate',)]
[('682966dc-f9d6-11ef-be00-c2c9fa6fd3d5', 3, 0.10425925999879837), ('682966dc-f9d6-11ef-be00-c2c9fa6fd3d5', 1, 0.15000000596046448), ('68296740-f9d6-11ef-be00-c2c9fa6fd3d5', 1, 1.2768545150756836), ('68296830-f9d6-11ef-be00-c2c9fa6fd3d5', 12, 0.11749999970197678), ('68296830-f9d6-11ef-be00-c2c9fa6fd3d5', 11, 0.10000000149011612), ('6829686c-f9d6-11ef-be00-c2c9fa6fd3d5', 2, 0.20333333313465118), ('682968a8-f9d6-11ef-be00-c2c9fa6fd3d5', 11, 0.13818182051181793), ('682968a8-f9d6-11ef-be00-c2c9fa6fd3d5', 9, 0.09358024597167969), ('682968c6-f9d6-11ef-be00-c2c9fa6fd3d5', 4, 0.059861112385988235), ('682968c6-f9d6-11ef-be00-c2c9fa6fd3d5', 6, 0.06421296298503876)]


/tmp/ipykernel_618/2164614495.py:2: DeprecationWarning: Python 3.9 support is deprecated and will be removed in a future release. This version of clickhouse-connect may stop working with Python 3.9 unexpectedly.
  from clickhouse_connect import get_client


В коде ниже приведён написанный Spark-скрипт. Ваша задача — правильно указать данные для подключения к вашему хранилищу: порты, параметры кластера ClickHouse и путь, куда будут записываться агрегаты.

In [ ]:
# filename=my_spark_job.py
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import sys

# Создаём Spark-сессию и при необходимости добавляем конфигурации
spark = SparkSession.builder.appName("myAggregateTest").config("fs.s3a.endpoint", "storage.yandexcloud.net").getOrCreate()

# Указываем порт и параметры кластера ClickHouse
jdbcPort = DATA_DELETED
jdbcHostname = "DATA_DELETED"
jdbcDatabase = "DATA_DELETED"
jdbcUrl = f"jdbc:clickhouse://{jdbcHostname}:{jdbcPort}/{jdbcDatabase}?ssl=true"


# Получаем аргумент из Airflow
my_date = sys.argv[1].replace('-', '_')

# Считываем исходные данные за нужную дату
df = spark.read.csv(f"s3a://da-plus-dags/script_bookmate/data_{my_date}/audition_content.csv", inferSchema=True, header=True)

# Строим агрегат по пользователям
result_df = df.groupBy("puid").agg(
    F.countDistinct("audition_id").alias("audition_count"),
    F.avg("hours").alias("avg_hours")
)

result_df.write.format("jdbc") \
    .option("url", jdbcUrl) \
    .option("user", "DATA_DELETED") \
    .option("password", "DATA_DELETED") \
    .option("dbtable", "DATA_DELETED") \
    .mode('append') \
    .save()

В результате будет создан файл с названием, указанным в первой строке. Этот файл можно будет запустить с помощью Airflow, но сначала понадобится настроить DAG.

## 2. Создание DAG

Теперь, когда Spark-код готов, нужно создать DAG, который будет его запускать.

### Задание 1

Создайте «каркас» нового DAG для вашего проекта. DAG должен запускаться каждый день начиная с 1 января 2025 года. При этом запускать DAG за пропущенные даты не нужно.

Используйте менеджер контекста `with ... as dag:` — так все задачи будут корректно привязаны к DAG. После конструкции пока напишите только `pass`.

In [ ]:
# filename=bookmate_dag.py

from datetime import datetime
from airflow import DAG
from airflow.sensors.s3_key_sensor import S3KeySensor
from airflow.providers.yandex.operators.dataproc import DataprocCreatePysparkJobOperator

class PysparkJobOperator(DataprocCreatePysparkJobOperator):
    template_fields = ("cluster_id", "args",)

DAG_ID = "audition_content_analysis"

with DAG(
    dag_id=DAG_ID,
    schedule_interval='@daily',
    start_date=datetime(2025, 1, 1),
    catchup=False,
    tags=['bookmate'],
) as dag:
    pass

### Задание 2

Теперь добавьте проверку входного файла. DAG не должен стартовать, пока в S3 не появится файл с данными за нужную дату.

Для решения используйте сенсор `S3KeySensor`. Он должен проверять наличие файла каждые 5 минут и ждать максимум час. В качестве аргумента для параметра `bucket_name` укажите строку `"da-plus-dags"`.

Файл называется `audition_content.csv`, но в имени папки должна быть дата запуска в формате `YYYY_MM_DD`. Например, для 5 января 2025 путь будет таким: `script_bookmate/data_2025_01_05/audition_content.csv`. Путь к данным  должен быть аргументом для параметра `bucket_key` в `S3KeySensor`.

In [ ]:
# filename=bookmate_dag.py

from datetime import datetime
from airflow import DAG
from airflow.sensors.s3_key_sensor import S3KeySensor
from airflow.providers.yandex.operators.dataproc import DataprocCreatePysparkJobOperator

class PysparkJobOperator(DataprocCreatePysparkJobOperator):
    template_fields = ("cluster_id", "args",)

DAG_ID = "audition_content_analysis"

wait_for_input = S3KeySensor(
        task_id="wait_for_input",
        poke_interval=300,
        timeout=3600,
        bucket_name="da-plus-dags",
        bucket_key="script_bookmate/data_{{ ds.replace('-', '_') }}/audition_content.csv",
        mode="poke",
        aws_conn_id="s3",
        wildcard_match=False,
    )

### Задание 3

Создайте задачу для запуска Spark-скрипта через Airflow. Укажите путь к файлу, который вы создали на первом шаге проекта.

In [ ]:
# filename=bookmate_dag.py

from airflow.providers.yandex.operators.dataproc import DataprocCreatePysparkJobOperator

run_pyspark = PysparkJobOperator(
        name="create_bookmate_aggregate",
        task_id="run_pyspark",
        cluster_id="c9q4134h5vi546h1e148",
        args=["{{ ds }}"],
        main_python_file_uri="s3a://da-plus-dags/DATA_DELETED/jobs/my_spark_job.py"
    )

### Задание 4

Теперь соберите все фрагменты вместе:

* Опишите DAG с нужными параметрами.

* Добавьте сенсор для ожидания входного файла.

* Добавьте Spark-задачу для запуска скрипта.

* Настройте зависимости так, чтобы Spark-задача запускалась только после появления файла.

Ваши данные для подключения к Airflow:
*   IP — DATA_DELETED
*   Имя пользователя — DATA_DELETED
*   Пароль — DATA_DELETED

In [ ]:
# filename=bookmate_dag.py

from datetime import datetime
from airflow import DAG
from airflow.sensors.s3_key_sensor import S3KeySensor
from airflow.providers.yandex.operators.dataproc import DataprocCreatePysparkJobOperator


class PysparkJobOperator(DataprocCreatePysparkJobOperator):
    template_fields = ("cluster_id", "args",)


DAG_ID = "audition_content_analysis"

with DAG(
    dag_id=DAG_ID,
    schedule="0 16 * * *",
    start_date=datetime(2025, 1, 1),
    catchup=False,
    tags=["bookmate"],
) as dag:

    # 1) Ждём появления входного файла в S3
    wait_for_input = S3KeySensor(
        task_id="wait_for_input",
        poke_interval=300,
        timeout=3600,
        bucket_name="da-plus-dags",
        bucket_key="script_bookmate/data_{{ ds.replace('-', '_') }}/audition_content.csv",
        mode="poke",
        aws_conn_id="s3",
        wildcard_match=False,
    )

    # 2) Запускаем PySpark-задание на кластере Dataproc
    run_pyspark = PysparkJobOperator(
        name="create_bookmate_aggregate",
        task_id="run_pyspark",
        cluster_id="c9q4134h5vi546h1e148",
        args=["{{ ds }}"],
        main_python_file_uri="s3a://da-plus-dags/DATA_DELETED/jobs/my_spark_job.py"
    )

    # 3) Зависимости
    wait_for_input >> run_pyspark

## 3. Запуск Airflow

Теперь можно переходить к запуску. Нажмите кнопку «Проверить», подождите 5 минут и снова нажмите её. Вам будут показаны данные для входа в веб-интерфейс Airflow. В интерфейсе найдите ваш DAG и запустите его.

Проверьте, что DAG выполнился, а результат соответствует ожиданиям. Если всё получилось — поздравляем, проект завершён!